In [ ]:

import pandas as pd

# Load MIMIC-IV demo patient data
patients = pd.read_csv("hosp/patients.csv.gz")

# Check the number of rows and columns
print("Dataset shape:", patients.shape)

# Show the first 5 rows
patients.head()


In [ ]:

# Load the hospital procedure records
procedures = pd.read_csv("hosp/procedures_icd.csv.gz")

# Show dataset size
print("Dataset shape:", procedures.shape)

# Show the first 5 rows
procedures.head()


In [ ]:

# Load procedure code descriptions
descriptions = pd.read_csv("hosp/d_icd_procedures.csv.gz")

# Find descriptions related to transfusions
transfusion_codes = descriptions[
    descriptions["long_title"].str.contains(
        "transfusion|packed red blood|red blood cells",
        case=False,
        na=False
    )
]

print("Matching procedure descriptions:", len(transfusion_codes))
transfusion_codes.head(20)


In [ ]:

# Match procedure records with transfusion code descriptions
matched = procedures.merge(
    transfusion_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Matching procedure records:", len(matched))
print("Number of patients:", matched["subject_id"].nunique())

matched[["subject_id", "icd_code", "long_title"]].head(10)


In [ ]:

# Find RBC-related procedure codes
rbc_codes = descriptions[
    descriptions["long_title"].str.contains(
        "red blood cell|packed red blood|packed cells",
        case=False,
        na=False
    )
]

print("RBC-related codes:", len(rbc_codes))

rbc_matched = procedures.merge(
    rbc_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

print("RBC-related procedure records:", len(rbc_matched))
print("Patients with RBC procedure codes:", rbc_matched["subject_id"].nunique())

rbc_matched[["subject_id", "icd_code", "long_title"]].head(10)


In [ ]:

# Load ICU item descriptions
items = pd.read_csv("icu/d_items.csv.gz")

# Find RBC or blood transfusion related items
blood_items = items[
    items["label"].str.contains(
        "red blood|packed red|transfusion|PRBC",
        case=False,
        na=False
    )
]

print("Blood-related ICU items:", len(blood_items))

blood_items[["itemid", "label", "category"]].head(20)


In [ ]:

# Load ICU input events
inputevents = pd.read_csv("icu/inputevents.csv.gz")

# Find packed red blood cell records
rbc_events = inputevents[
    inputevents["itemid"].isin([225168, 220996])
]

print("Total input events:", len(inputevents))
print("RBC input records:", len(rbc_events))
print("Patients with RBC records:", rbc_events["subject_id"].nunique())

rbc_events.head()


In [ ]:

# Check which RBC items were recorded
print("RBC item IDs:")
print(rbc_events["itemid"].value_counts())

# Check order status
print("\nRecord status:")
print(rbc_events["statusdescription"].value_counts(dropna=False))

# Show a few RBC records
rbc_events[
    ["subject_id", "itemid", "amount", "amountuom", "statusdescription"]
].head(10)


In [ ]:

# Find procedure records for patients with RBC input events
rbc_patient_ids = rbc_events["subject_id"].unique()

patient_procedures = procedures[
    procedures["subject_id"].isin(rbc_patient_ids)
]

# Add descriptions for their procedures
patient_procedures = patient_procedures.merge(
    descriptions,
    on=["icd_code", "icd_version"],
    how="left"
)

print("Procedure records:", len(patient_procedures))

patient_procedures[
    ["subject_id", "icd_code", "long_title"]
].head(20)


In [ ]:

# Find heart surgery-related procedure descriptions
heart_surgery = patient_procedures[
    patient_procedures["long_title"].str.contains(
        "coronary artery bypass|coronary bypass|open heart surgery",
        case=False,
        na=False
    )
]

# Admissions with RBC-related ICU records
rbc_admissions = rbc_events[
    ["subject_id", "hadm_id"]
].drop_duplicates()

# Match heart procedures and RBC records from the same admission
same_admission = heart_surgery.merge(
    rbc_admissions,
    on=["subject_id", "hadm_id"],
    how="inner"
)

print("Matching heart-related procedure records:", len(same_admission))
print("Patients:", same_admission["subject_id"].nunique())
print("Admissions:", same_admission["hadm_id"].nunique())

same_admission[
    ["subject_id", "hadm_id", "icd_code", "long_title"]
].head(10)


In [ ]:

# Keep one row per admission and procedure date
surgery_dates = same_admission[
    ["subject_id", "hadm_id", "chartdate"]
].drop_duplicates()

# Get RBC input event dates
rbc_dates = rbc_events[
    ["subject_id", "hadm_id", "starttime"]
].copy()

# Convert to dates
surgery_dates["chartdate"] = pd.to_datetime(surgery_dates["chartdate"])
rbc_dates["starttime"] = pd.to_datetime(rbc_dates["starttime"])

# First RBC record in each admission
first_rbc = (
    rbc_dates.groupby(["subject_id", "hadm_id"])["starttime"]
    .min()
    .reset_index()
)

# Compare procedure date with first RBC record
comparison = surgery_dates.merge(
    first_rbc,
    on=["subject_id", "hadm_id"]
)

comparison["days_difference"] = (
    comparison["starttime"].dt.normalize()
    - comparison["chartdate"]
).dt.days

comparison[
    ["subject_id", "chartdate", "starttime", "days_difference"]
].head(15)
